# Alpha 派最新资讯调试 Notebook

这个 notebook 不直接重写爬虫逻辑，而是调用项目里的 Node/Playwright 脚本，方便你逐步调试：登录、探测个股页、查看候选资讯、运行单个股票提取。

In [ ]:
from pathlib import Path
import json
import os
import subprocess

PROJECT = Path(r"E:\\策略\\web-auto")
assert PROJECT.exists(), PROJECT
PROJECT

## 1. 检查 Node 和依赖

In [ ]:
def run(cmd, env=None, timeout=180):
    full_env = os.environ.copy()
    if env:
        full_env.update(env)
    result = subprocess.run(
        cmd,
        cwd=PROJECT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
        shell=True,
        env=full_env,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return result.stdout

run("node -v")
run("node --check src\\latest-news\\extract.js")
run("node --check src\\latest-news\\probe.js")

## 2. 查看配置和股票清单

In [ ]:
config = json.loads((PROJECT / "config" / "latest-extraction.json").read_text(encoding="utf-8"))
print(json.dumps(config, ensure_ascii=False, indent=2))
print("\n--- data/stocks.csv ---")
print((PROJECT / "data" / "stocks.csv").read_text(encoding="utf-8"))

## 3. 登录

如果 `runtime/auth.json` 已经存在且没过期，可以跳过。需要重新登录时，运行下一格，会弹出 Edge 窗口。

In [ ]:
# 手动登录：弹出 Edge，登录成功后保存 runtime/auth.json
# run("npm run login", timeout=360)

# 自动登录：不要把真实密码保存进 notebook，运行前临时填入或用系统环境变量
# run("npm run login:auto", env={"ALPHA_PHONE": "你的手机号", "ALPHA_PASSWORD": "你的密码"}, timeout=180)

auth = PROJECT / "runtime" / "auth.json"
print("auth file exists:", auth.exists(), "size:", auth.stat().st_size if auth.exists() else 0)
print("\nReal login check:")
try:
    run("npm run login:check", timeout=120)
except Exception as exc:
    print(exc)

## 4. 探测单个股票页面

`probe` 不做正式导出，只保存页面截图和候选资讯 JSON。改爬虫前建议先跑它。

In [ ]:
code = "AVAV.US"
name = "AeroVironment,Inc."
run(f"npm run probe -- --code {code} --name {name}", timeout=180)

debug_json = PROJECT / "output" / "debug" / f"{code}_{name}.json"
data = json.loads(debug_json.read_text(encoding="utf-8"))
data["url"], data["title"], len(data["candidates"])


In [ ]:
for item in data["candidates"][:10]:
    print(
        item["index"],
        item.get("dateText"),
        item.get("type"),
        item.get("typeName"),
        item.get("institutionText"),
        item.get("title") or "<summary/comment>",
    )


## 5. 跑正式提取

默认读取 `data/stocks.csv`。调试时建议 CSV 里只保留一个股票。

In [ ]:
run("npm run extract", timeout=240)

out_dir = PROJECT / "output" / "latest-news"
print(out_dir)
for path in sorted(out_dir.rglob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)[:5]:
    print(path)

## 6. 查看最新 TXT 正文

In [ ]:
txt_files = sorted((PROJECT / "output" / "latest-news").rglob("*.txt"), key=lambda p: p.stat().st_mtime, reverse=True)
latest_txt = txt_files[0]
print(latest_txt)
print(latest_txt.read_text(encoding="utf-8")[:2000])